# Descripción del proyecto

La compañía Sweet Lift Taxi ha recopilado datos históricos sobre pedidos de taxis en los aeropuertos. Para atraer a más conductores durante las horas pico, necesitamos predecir la cantidad de pedidos de taxis para la próxima hora. Construye un modelo para dicha predicción.

La métrica RECM en el conjunto de prueba no debe ser superior a 48.

## Instrucciones del proyecto.

1. Descarga los datos y haz el remuestreo por una hora.
2. Analiza los datos
3. Entrena diferentes modelos con diferentes hiperparámetros. La muestra de prueba debe ser el 10% del conjunto de datos inicial.4. Prueba los datos usando la muestra de prueba y proporciona una conclusión.

## Descripción de los datos

Los datos se almacenan en el archivo `taxi.csv`. 	
El número de pedidos está en la columna `num_orders`.

## Preparación

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


##### Carga de datos


In [2]:
data = pd.read_csv('/datasets/taxi.csv', index_col=[0], parse_dates=[0])
data.sort_index(inplace=True)

##### Remuestreo a la hora 1 


In [3]:
data = data.resample('1H').sum()

print(data.head())
print(data.info())


                     num_orders
datetime                       
2018-03-01 00:00:00         124
2018-03-01 01:00:00          85
2018-03-01 02:00:00          71
2018-03-01 03:00:00          66
2018-03-01 04:00:00          43
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4416 entries, 2018-03-01 00:00:00 to 2018-08-31 23:00:00
Freq: H
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   num_orders  4416 non-null   int64
dtypes: int64(1)
memory usage: 69.0 KB
None


## Análisis

data.plot(figsize=(12,5), title="Pedidos por hora")
plt.show()
print(data.describe())


##### Ingenieria de caracteristicas

In [4]:
def make_features(data, max_lag, rolling_mean_size):
    data['year'] = data.index.year
    data['month'] = data.index.month
    data['day'] = data.index.day
    data['dayofweek'] = data.index.dayofweek
    data['hour'] = data.index.hour

    for lag in range(1, max_lag + 1):
        data[f'lag_{lag}'] = data['num_orders'].shift(lag)

    data['rolling_mean'] = (
        data['num_orders'].shift().rolling(rolling_mean_size).mean()
    )
make_features(data, max_lag=24, rolling_mean_size=24)

data = data.dropna()

## Formación

##### Formacion de modelos, split temporal

In [5]:
train, test = train_test_split(data, shuffle=False, test_size=0.1)

features = [col for col in data.columns if col != 'num_orders']
target = 'num_orders'

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

##### Modelo 1, modelo de regresion lineal

In [6]:
lr = LinearRegression()
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

rmse_lr = mean_squared_error(y_test, pred_lr, squared=False)
print("RMSE Linear Regression:", rmse_lr)


RMSE Linear Regression: 45.83447405433358


##### Modelo , modelo de arbol


In [7]:
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

rmse_rf = mean_squared_error(y_test, pred_rf, squared=False)
print("RMSE Random Forest:", rmse_rf)

RMSE Random Forest: 43.60872020461009


## Prueba

In [8]:
print("\nRESULTADOS FINALES:")
print("Linear Regression RMSE:", rmse_lr)
print("Random Forest RMSE:", rmse_rf)

if rmse_rf < 48:
    print("El modelo cumple con el requisito")
else:
    print("Ajustar hiperparámetros")


RESULTADOS FINALES:
Linear Regression RMSE: 45.83447405433358
Random Forest RMSE: 43.60872020461009
El modelo cumple con el requisito


# Lista de revisión

- [x]  	
Jupyter Notebook está abierto.
- [x]  El código no tiene errores
- [x]  Las celdas con el código han sido colocadas en el orden de ejecución.
- [x]  	
Los datos han sido descargados y preparados.
- [x]  Se ha realizado el paso 2: los datos han sido analizados
- [x]  Se entrenó el modelo y se seleccionaron los hiperparámetros
- [x]  Se han evaluado los modelos. Se expuso una conclusión
- [x] La *RECM* para el conjunto de prueba no es más de 48